# Estimation error versus uniform-noise amplitude

This notebook studies how the least-squares and constrained weighted-LASSO estimation errors vary with the uniform-noise amplitude $\mu$ at a fixed sample size. The large regression statistics are precomputed once per Monte Carlo realization and reused over the full noise sweep. Numba is used when installed; otherwise the notebook falls back automatically to the same Python implementation.

### Functions and classes

| Function / class | Main inputs | Purpose |
|---|---|---|
| `design_observer` | `A, C` | Place the observer poles and return the observer gain and observer state matrix. |
| `compute_true_theta` | `Ao, B, C, G, n` | Compute the true observer parameter vector used as the estimation reference. |
| `make_output_filter` | `A, B, C` | Convert the state-space system to the discrete transfer-function coefficients used by `lfilter`. |
| `epsilon_star` | `mu, theta_true` | Compute the theoretical residual bound for each uniform-noise amplitude. |
| `adaptive_weights` | `theta` | Build normalized adaptive weights from a parameter estimate. |
| `gram_poly_blocks` | `u, yc, z, nblocks` | Accumulate polynomial coefficients of the regression statistics; Numba accelerates this function when available. |
| `precompute_statistics` | `u, yc, z` | Combine the blockwise Numba outputs into the regression statistics used for the full mu sweep. |
| `ls_statistics_at_mu` | `poly, mu` | Evaluate the regression statistics at a selected noise amplitude and compute the LS estimate. |
| `psd_factor` | `Q` | Compute a square-root factor of a positive-semidefinite matrix for the residual constraint. |
| `WeightedLassoSolver` | `p` | Reusable CVXPY solver for the constrained weighted-LASSO problem. |
| `run_mu_sweep` | `` | Run the Monte Carlo sweep over the configured noise amplitudes and collect LS/LASSO errors. |
| `plot_results` | `r` | Plot the mean estimation errors and standard-deviation bands versus the noise amplitude. |
| `save_figure` | `fig` | Save the figure as PDF/PNG and, when available, TikZ/LaTeX. |


In [ ]:
from pathlib import Path
import numpy as np
from scipy.signal import lfilter, place_poles, ss2tf
try:
    from numba import njit, prange
    NUMBA_AVAILABLE = True
except ImportError:
    NUMBA_AVAILABLE = False
    njit = None
    prange = range
import matplotlib.pyplot as plt
import cvxpy as cp

# System / settings
A=np.array([[.7,.1],[.2,.8]],float)
B=np.array([[1.],[1.]])
C=np.array([[1.,-1.]])

FIXED_N=1_000_000
N_LAG=2
NUM_TRIALS=100
MU_VALUES=np.logspace(-5,2,36)

WEIGHT_DELTA=1e-6
WEIGHT_POWER=1.0
LAMBDA_L2=0.0
RANDOM_SEED=555
NUMBA_BLOCKS=256

FIG_DIR=Path("Figures")
FIG_BASENAME="ls_lasso_uniform_error_vs_mu_N1million"


# Observer / true theta
def design_observer(A,C):
    poles=np.array([1e-8,2e-15])
    obj=place_poles(A.T,C.T,poles)
    G=obj.gain_matrix.T
    Ao=A-G@C
    print("Desired poles:",poles)
    print("Actual poles:",np.linalg.eigvals(Ao))
    return G,Ao

def compute_true_theta(Ao,B,C,G,n):
    tu=np.empty(n); ty=np.empty(n)
    AB=B.copy(); AG=G.copy()
    for i in range(n):
        tu[i]=(C@AB).item(); ty[i]=(C@AG).item()
        AB=Ao@AB; AG=Ao@AG
    return np.hstack((tu,ty))

def make_output_filter(A,B,C):
    num,den=ss2tf(A,B,C,np.zeros((1,1)))
    return np.asarray(num[0],float),np.asarray(den,float)

def epsilon_star(mu,theta_true):
    ty=theta_true[N_LAG:]
    return float(mu)/np.sqrt(3.0)*np.sqrt(1.0+ty@ty)

def adaptive_weights(theta):
    w=1.0/(np.abs(theta)+WEIGHT_DELTA)**WEIGHT_POWER
    return w/w.mean()


# Precomputed polynomial statistics
#
# x(mu) = a + mu*b
# Y(mu) = c + mu*d
#
# Therefore:
# Q(mu)  = Q0 + mu Q1 + mu^2 Q2
# q(mu)  = q0 + mu q1 + mu^2 q2
# Y'Y(mu)= s0 + mu s1 + mu^2 s2
#
# These coefficients are computed once per trial.
def _gram_poly_blocks_python(u, yc, z, nblocks):
        Q0=np.zeros((nblocks,4,4)); Q1=np.zeros((nblocks,4,4)); Q2=np.zeros((nblocks,4,4))
        q0=np.zeros((nblocks,4)); q1=np.zeros((nblocks,4)); q2=np.zeros((nblocks,4))
        ss=np.zeros((nblocks,3))
        n=u.size

        for bb in prange(nblocks):
            lo=2+(n-2)*bb//nblocks
            hi=2+(n-2)*(bb+1)//nblocks

            g0=np.zeros((4,4)); g1=np.zeros((4,4)); g2=np.zeros((4,4))
            h0=np.zeros(4); h1=np.zeros(4); h2=np.zeros(4)
            r0=0.0; r1=0.0; r2=0.0
            a=np.empty(4); b=np.empty(4)

            for k in range(lo,hi):
                a[0]=u[k-1]; a[1]=u[k-2]
                a[2]=yc[k-1]; a[3]=yc[k-2]

                b[0]=0.0; b[1]=0.0
                b[2]=z[k-1]; b[3]=z[k-2]

                c=yc[k]; d=z[k]

                for i in range(4):
                    h0[i]+=a[i]*c
                    h1[i]+=a[i]*d+b[i]*c
                    h2[i]+=b[i]*d

                    for j in range(4):
                        g0[i,j]+=a[i]*a[j]
                        g1[i,j]+=a[i]*b[j]+b[i]*a[j]
                        g2[i,j]+=b[i]*b[j]

                r0+=c*c
                r1+=2.0*c*d
                r2+=d*d

            for i in range(4):
                q0[bb,i]=h0[i]; q1[bb,i]=h1[i]; q2[bb,i]=h2[i]
                for j in range(4):
                    Q0[bb,i,j]=g0[i,j]
                    Q1[bb,i,j]=g1[i,j]
                    Q2[bb,i,j]=g2[i,j]

            ss[bb,0]=r0; ss[bb,1]=r1; ss[bb,2]=r2

        return Q0,Q1,Q2,q0,q1,q2,ss

if NUMBA_AVAILABLE:
    gram_poly_blocks = njit(cache=True, parallel=True)(_gram_poly_blocks_python)
else:
    gram_poly_blocks = _gram_poly_blocks_python


def precompute_statistics(u,yc,z):
    out=gram_poly_blocks(u,yc,z,NUMBA_BLOCKS)
    Q0,Q1,Q2=[x.sum(axis=0,dtype=np.longdouble) for x in out[:3]]
    q0,q1,q2=[x.sum(axis=0,dtype=np.longdouble) for x in out[3:6]]
    s=out[6].sum(axis=0,dtype=np.longdouble)
    return Q0,Q1,Q2,q0,q1,q2,s


def ls_statistics_at_mu(poly,mu):
    Q0,Q1,Q2,q0,q1,q2,s=poly
    m=np.longdouble(mu); m2=m*m

    Qld=Q0+m*Q1+m2*Q2
    qld=q0+m*q1+m2*q2
    y2=s[0]+m*s[1]+m2*s[2]

    Q=np.asarray(Qld,dtype=np.float64)
    q=np.asarray(qld,dtype=np.float64)
    Q=.5*(Q+Q.T)

    try: theta=np.linalg.solve(Q,q)
    except np.linalg.LinAlgError: theta=np.linalg.lstsq(Q,q,rcond=None)[0]

    tld=theta.astype(np.longdouble)
    rss=y2-2.0*(tld@qld)+tld@Qld@tld
    return theta,Q,max(float(rss),0.0)


# Constrained weighted LASSO
def psd_factor(Q):
    Q=.5*(Q+Q.T)
    try: return np.linalg.cholesky(Q).T
    except np.linalg.LinAlgError:
        d,V=np.linalg.eigh(Q)
        d=np.clip(d,0,None)
        return np.sqrt(d)[:,None]*V.T


class WeightedLassoSolver:
    def __init__(self,p):
        self.theta=cp.Variable(p)
        self.R=cp.Parameter((p,p))
        self.target=cp.Parameter(p)
        self.radius=cp.Parameter(nonneg=True)
        self.w=cp.Parameter(p,nonneg=True)

        obj=cp.norm1(cp.multiply(self.w,self.theta))
        if LAMBDA_L2>0: obj+=.5*LAMBDA_L2*cp.sum_squares(self.theta)

        self.problem=cp.Problem(cp.Minimize(obj),[
            cp.norm(self.R@self.theta-self.target,2)<=self.radius
        ])
        self.solvers=set(cp.installed_solvers())

    def solve(self,Q,theta_ls,rss,T,epsilon,w):
        # Normalize by T: mathematically identical, numerically better.
        Qn=Q/T
        r2=epsilon**2-rss/T
        tol=1e-12*max(1.0,epsilon**2,rss/T)

        if r2 < -tol:
            raise ValueError(
                f"infeasible: eps={epsilon:.3e}, "
                f"minimum RMS={np.sqrt(rss/T):.3e}"
            )

        R=psd_factor(Qn)
        self.R.value=R
        self.target.value=R@theta_ls
        self.radius.value=np.sqrt(max(r2,0.0))
        self.w.value=w

        if "CLARABEL" in self.solvers:
            try:
                self.problem.solve(
                    solver="CLARABEL",warm_start=True,verbose=False,
                    max_iter=500,tol_gap_abs=1e-8,
                    tol_gap_rel=1e-8,tol_feas=1e-8
                )
                if self.theta.value is not None:
                    return np.asarray(self.theta.value,float).ravel()
            except Exception: pass

        if "ECOS" in self.solvers:
            try:
                self.problem.solve(
                    solver="ECOS",warm_start=True,verbose=False,
                    max_iters=2000,reltol=1e-7,
                    abstol=1e-7,feastol=1e-7
                )
                if self.theta.value is not None:
                    return np.asarray(self.theta.value,float).ravel()
            except Exception: pass

        self.problem.solve(
            solver="SCS",warm_start=True,verbose=False,
            eps=1e-5,max_iters=10000
        )
        if self.theta.value is None: raise RuntimeError("LASSO failed")
        return np.asarray(self.theta.value,float).ravel()


# Experiment
def run_mu_sweep():
    G,Ao=design_observer(A,C)
    theta_true=compute_true_theta(Ao,B,C,G,N_LAG)
    num,den=make_output_filter(A,B,C)

    mu_values=np.asarray(MU_VALUES,float)
    eps_values=np.array([epsilon_star(mu,theta_true) for mu in mu_values])

    ls_err=np.full((len(mu_values),NUM_TRIALS),np.nan)
    la_err=np.full_like(ls_err,np.nan)

    solver=WeightedLassoSolver(2*N_LAG)
    rng=np.random.default_rng(RANDOM_SEED)

    print("theta_true =",theta_true)
    print("||theta_true|| =",np.linalg.norm(theta_true))
    print("epsilon*/mu =",eps_values[0]/mu_values[0])
    print("N =",f"{FIXED_N:,}")
    print("Numba acceleration:", "enabled" if NUMBA_AVAILABLE else "not available; using Python fallback")

    # Trigger Numba compilation before the Monte Carlo loop when available.
    if NUMBA_AVAILABLE:
        _u=np.zeros(10); _y=np.zeros(10); _z=np.zeros(10)
        gram_poly_blocks(_u,_y,_z,1)

    for trial in range(NUM_TRIALS):
        u=rng.standard_normal(FIXED_N)
        z=rng.uniform(-1.0,1.0,FIXED_N)
        y_clean=lfilter(num,den,u)

        # Precompute the large-data statistics once for this trial.
        poly=precompute_statistics(u,y_clean,z)
        T=FIXED_N-N_LAG

        for j,mu in enumerate(mu_values):
            theta_ls,Q,rss=ls_statistics_at_mu(poly,mu)
            ls_err[j,trial]=np.linalg.norm(theta_ls-theta_true)

            try:
                theta_la=solver.solve(
                    Q,theta_ls,rss,T,eps_values[j],
                    adaptive_weights(theta_ls)
                )
                la_err[j,trial]=np.linalg.norm(theta_la-theta_true)
            except Exception as exc:
                print(f"LASSO fail trial={trial+1}, mu={mu:.3e}: {exc}")

        print(f"Finished trial {trial+1}/{NUM_TRIALS}")

    return dict(
        N=FIXED_N,theta_true=theta_true,
        mu_values=mu_values,epsilon_values=eps_values,
        ls_errors=ls_err,lasso_errors=la_err,
        ls_mean=np.nanmean(ls_err,axis=1),
        ls_std=np.nanstd(ls_err,axis=1),
        lasso_mean=np.nanmean(la_err,axis=1),
        lasso_std=np.nanstd(la_err,axis=1)
    )


# Plot / save
def plot_results(r):
    mu=r["mu_values"]; tiny=np.finfo(float).tiny
    fig,ax=plt.subplots(figsize=(7.6,4.8))

    ax.loglog(mu,r["ls_mean"],"o-",lw=2,ms=5,label="LS")
    ax.fill_between(
        mu,np.maximum(r["ls_mean"]-r["ls_std"],tiny),
        r["ls_mean"]+r["ls_std"],alpha=.10
    )

    ax.loglog(
        mu,r["lasso_mean"],"s--",lw=2,ms=5,
        label=r"Weighted LASSO, $\epsilon=\epsilon^\star$"
    )
    ax.fill_between(
        mu,np.maximum(r["lasso_mean"]-r["lasso_std"],tiny),
        r["lasso_mean"]+r["lasso_std"],alpha=.10
    )

    ax.set_xlabel(r"Uniform-noise amplitude $\mu$")
    ax.set_ylabel(r"Estimation error $\|\hat{\theta}-\theta^\star\|_2$")
    ax.grid(True,which="both",ls="--",lw=.6,alpha=.6)
    ax.legend()
    fig.tight_layout()
    return fig


def save_figure(fig):
    FIG_DIR.mkdir(parents=True,exist_ok=True)
    pdf=FIG_DIR/f"{FIG_BASENAME}.pdf"
    png=FIG_DIR/f"{FIG_BASENAME}.png"
    tex=FIG_DIR/f"{FIG_BASENAME}.tex"

    fig.savefig(pdf,bbox_inches="tight")
    fig.savefig(png,dpi=300,bbox_inches="tight")

    try:
        import tikzplotlib

        for ax in fig.axes:
            for line in ax.get_lines():
                if not hasattr(line,"_us_dashSeq"):
                    p=getattr(line,"_unscaled_dash_pattern",None)
                    if p is None: p=getattr(line,"_dash_pattern",None)
                    off,seq=(0,()) if p is None else p
                    line._us_dashSeq=() if seq is None else seq
                    line._us_dashOffset=off
            leg=ax.get_legend()
            if leg is not None and hasattr(leg,"_ncols") and not hasattr(leg,"_ncol"):
                leg._ncol=leg._ncols

        tikzplotlib.save(
            str(tex),figure=fig,
            axis_width=r"\linewidth",
            axis_height=r"0.60\linewidth",
            strict=False,standalone=False
        )
        print("Saved TikZ:",tex)
    except Exception as exc:
        print("TikZ save skipped:",exc)

    print("Saved PDF:",pdf)
    print("Saved PNG:",png)


### Main experiment

This block runs the configured Monte Carlo experiment, saves the numerical results, generates the error-versus-noise figure, and prints the summary table.


In [ ]:
def main():
    r=run_mu_sweep()
    FIG_DIR.mkdir(parents=True,exist_ok=True)
    np.savez(FIG_DIR/f"{FIG_BASENAME}_results.npz",**r)

    fig=plot_results(r)
    save_figure(fig)

    print("\nmu          epsilon*      LS mean       LASSO mean")
    for mu,eps,ls,la in zip(
        r["mu_values"],r["epsilon_values"],
        r["ls_mean"],r["lasso_mean"]
    ):
        print(f"{mu:10.3e}  {eps:10.3e}  {ls:12.5e}  {la:12.5e}")

    plt.show()
    return r


if __name__=="__main__":
    results_mu=main()
